In [42]:
from PIL import Image
def Brezenheim_otrezok(img, x0, y0, x1, y1):
    b = int((img.size[0] - 1) / 2)
    pixels = img.load()
    steep = abs(y1 - y0) > abs(x1 - x0)
    if steep:
        x0, y0 = y0, x0
        x1, y1 = y1, x1
    if x0 > x1:
        x0, x1 = x1, x0
        y0, y1 = y1, y0
    dx = x1 - x0
    dy = abs(y1 - y0)
    d = 2 * dy - dx
    y = y0
    step_y = 1 if y1 > y0 else -1
    for x in range(x0, x1 + 1):
        curr_x, curr_y = (y, x) if steep else (x, y)
        pixels[b + curr_x, b - curr_y] = 0
        if d > 0:
            y += step_y
            d -= 2 * dx
        d += 2 * dy

6.      Реализовать двумерные алгоритмы отсечения отрезков прямоугольным окном и окном, представляющим собой выпуклый многоугольник: Коэна-Сазерленда, Лианга-Барски, Кируса-Бека.

Алгоритм Коэна-Сазерленда

In [5]:
inside = 0
left = 1
right = 2
bottom = 4
top = 8


def compute_code(x, y, x_min, y_min, x_max, y_max):
    code = inside
    if x < x_min:
        code |= left
    if x > x_max:
        code |= right
    if y < y_min:
        code |= bottom
    if y > y_max:
        code |= top
    return code


def alg_1(x0, y0, x1, y1, x_min, y_min, x_max, y_max):
    code0 = compute_code(x0, y0, x_min, y_min, x_max, y_max)
    code1 = compute_code(x1, y1, x_min, y_min, x_max, y_max)
    flag = False

    def cutter(x, y, code):
        if code & left != 0:
            y = y + (x_min - x) * (y1 - y0) / (x1 - x0)
            x = x_min
        elif code & right != 0:
            y = y + (x_max - x) * (y1 - y0) / (x1 - x0)
            x = x_max
        elif code & bottom != 0:
            x = x + (y_min - y) * (x1 - x0) / (y1 - y0)
            y = y_min
        elif code & top != 0:
            x = x + (y_max - y) * (x1 - x0) / (y1 - y0)
            y = y_max
        return x, y

    while True:
        if code0 | code1 == 0:
            flag = True
            break
        elif code0 & code1 != 0:
            break
        else:
            if code0 != 0:
                x0, y0 = cutter(x0, y0, code0)
                code0 = compute_code(x0, y0, x_min, y_min, x_max, y_max)
            elif code1 != 0:
                x1, y1 = cutter(x1, y1, code1)
                code1 = compute_code(x1, y1, x_min, y_min, x_max, y_max)

    if flag:
        return x0, y0, x1, y1

In [47]:
img = Image.new("L", (41, 41), color=255)
x0, y0, x1, y1 = alg_1(100, 80, -100, -80, -20, -20, 20, 20)
Brezenheim_otrezok(img, x0, y0, x1, y1)
img.show()

In [ ]:
Алгоритм Лианга-Барски

In [53]:
def alg_2(x0, y0, x1, y1, x_min, y_min, x_max, y_max):
    dx = x1 - x0
    dy = y1 - y0
    p = [-dx, dx, -dy, dy]
    q = [x0 - x_min, x_max - x0, y0 - y_min, y_max - y0]
    t0, t1 = 0.0, 1.0
    for i in range(4):
        if p[i] == 0:
            if q[i] < 0:
                return None
        else:
            t = q[i] / p[i]
            if p[i] < 0:
                t0 = max(t0, t)
            else:
                t1 = min(t1, t)
    if t0 > t1:
        return None
    return map(round, [x0 + t0 * dx, y0 + t0 * dy, x0 + t1 * dx, y0 + t1 * dy])

In [56]:
img = Image.new("L", (41, 41), color=255)
x0, y0, x1, y1 = alg_2(100, 80, -100, -80, -20, -20, 20, 20)
Brezenheim_otrezok(img, x0, y0, x1, y1)
img.show()

Алгоритм Кируса-Бека.

In [68]:
def alg_3(x0, y0, x1, y1, vertices, face):
    D_x = x1 - x0
    D_y = y1 - y0
    t_enter = 0.0
    t_leave = 1.0
    n = len(face)
    for i in range(n):
        idx_curr = face[i]
        idx_next = face[(i + 1) % n]
        p_edge1 = vertices[idx_curr]
        p_edge2 = vertices[idx_next]
        edge_dx = p_edge2[0] - p_edge1[0]
        edge_dy = p_edge2[1] - p_edge1[1]
        N_x = -edge_dy
        N_y = edge_dx
        W_x = x0 - p_edge1[0]
        W_y = y0 - p_edge1[1]
        P_i = D_x * N_x + D_y * N_y
        Q_i = W_x * N_x + W_y * N_y
        if P_i == 0:
            if Q_i < 0:
                return None
        else:
            t = -Q_i / P_i
            if P_i > 0:
                t_enter = max(t_enter, t)
            else:
                t_leave = min(t_leave, t)
    if t_enter > t_leave:
        return None
    x_start_visible = x0 + t_enter * D_x
    y_start_visible = y0 + t_enter * D_y
    x_end_visible = x0 + t_leave * D_x
    y_end_visible = y0 + t_leave * D_y
    return map(round, [x_start_visible, y_start_visible, x_end_visible, y_end_visible])

In [72]:
img = Image.new("L", (41, 41), color=255)
vertices = [[-20, -20], [20, 20], [-20, 20]]
face = [0, 1, 2]
x0, y0, x1, y1 = alg_3(100, 80, -100, -80, vertices, face)
Brezenheim_otrezok(img, x0, y0, x1, y1)
img.show()